In [1]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import tqdm
from scipy.stats import poisson

In [2]:
df_matches = pd.read_pickle("data/df_matches_2010_2024.pickle")
n_teams = df_matches["host_id"].nunique()
team_to_idx = {team: idx for idx, team in enumerate(df_matches["host_name"].unique())}

In [3]:
def prepare_df_long(df_matches: pd.DataFrame):
    df_home = df_matches.copy().rename(columns={
        "host_name": "team_name",
        "host_goals": "goals",
        "guest_name": "opponent_name",
        "guest_goals": "opponent_goals"
    })

    df_away = df_matches.copy().rename(columns={
        "host_name": "opponent_name",
        "host_goals": "opponent_goals",
        "guest_name": "team_name",
        "guest_goals": "goals"
    })

    df_home["home"] = 1.0
    df_away["home"] = 0.0

    return pd.concat([df_home, df_away], ignore_index=True)


In [4]:
class Prediction:
    def __init__(self, p_scorelines) -> None:
        self.p_scorelines = p_scorelines

    def max_util_scoreline(self):
        utilities = self._expected_utilities()
        max_index = np.unravel_index(np.argmax(utilities), utilities.shape)
        return tuple(int(x) for x in max_index)

    def plot_joint_pred(self):
        plt.figure()
        sns.heatmap(self.p_scorelines, annot=True, fmt=".2f", cmap="YlGnBu")
        plt.title("Joint Probability Distribution of Predicted Scorelines")
        plt.xlabel("Away Team Goals")
        plt.ylabel("Home Team Goals")
        plt.show()
    
    def plot_exp_utilities(self):
        utilities = self._expected_utilities()
        plt.figure()
        sns.heatmap(utilities, annot=True, fmt=".2f", cmap="YlGnBu")
        plt.title("Expected Utilities")
        plt.xlabel("Away Team Goals")
        plt.ylabel("Home Team Goals")
        plt.show()
    
    @staticmethod
    def plot_score_rule(correct_result, max_goals=4):
        x = np.arange(0, max_goals + 1)
        y = [Prediction._scoring_rule((i, j), correct_result) for i in x for j in x]
        sns.heatmap(np.array(y).reshape(max_goals + 1, max_goals + 1), annot=True, fmt=".2f", cmap="YlGnBu")
        plt.title("Scoring Rule Heatmap")
        plt.xlabel("Predicted Away Goals")
        plt.ylabel("Predicted Home Goals")
        plt.show()

    def _expected_utilities(self):
        utilities = np.zeros_like(self.p_scorelines)
        for i in range(self.p_scorelines.shape[0]):
            for j in range(self.p_scorelines.shape[1]):
                utilities[i, j] = self._expected_utility((i, j))
        return utilities
    
    def _expected_utility(self, ground_truth):
        ep = 0
        for i in range(self.p_scorelines.shape[0]):
            for j in range(self.p_scorelines.shape[1]):
                ep += Prediction._scoring_rule((i, j), ground_truth) * self.p_scorelines[i, j]
        return ep

    @staticmethod
    def _scoring_rule(pred, ground_truth):
        home_pred, away_pred = pred
        home_gt, away_gt = ground_truth

        if home_pred == home_gt and away_pred == away_gt:
            return 4 # correct result
        if home_pred - away_pred == home_gt - away_gt:
            return 2 if home_pred == away_pred else 3 # correct difference
        if home_pred > away_pred and home_gt > away_gt or home_pred < away_pred and home_gt < away_gt:
            return 2 # correct trend
        return 0 # incorrect

assert Prediction._scoring_rule((1, 0), (1, 0)) == 4 # correct
assert Prediction._scoring_rule((1, 0), (2, 0)) == 2 # correct trend
assert Prediction._scoring_rule((1, 0), (0, 0)) == 0 # incorrect
assert Prediction._scoring_rule((2, 2), (1, 1)) == 2 # correct difference (draw)
assert Prediction._scoring_rule((3, 2), (2, 1)) == 3 # correct difference (no draw)

class PoissonModel:
    def fit(self, df_long):
        self.model = smf.glm(
            formula="goals ~ home + team_name + opponent_name",
            data=df_long,
            family=sm.families.Poisson()
        ).fit()
    
    def predict(self, home_name, guest_name, max_goals=4):                
        new_match = pd.DataFrame({
            'team_name': [home_name, guest_name],
            'opponent_name': [guest_name, home_name],
            'home': [1, 0]
        })
        try:
            home_exp, away_exp = self.model.predict(new_match)
        except Exception as e:
            print(f"Error predicting match: {e}")
            home_exp, away_exp = 0, 0

        home_goals_prob = [poisson.pmf(i, home_exp) for i in range(max_goals)]
        away_goals_prob = [poisson.pmf(i, away_exp) for i in range(max_goals)]

        # sum tails
        home_goals_prob.append(1 - np.sum(home_goals_prob))
        away_goals_prob.append(1 - np.sum(away_goals_prob))

        p_scorelines = np.outer(home_goals_prob, away_goals_prob)

        return Prediction(p_scorelines)

In [10]:
df_matches_all = pd.read_pickle("data/df_matches_all.pickle")
horizons = range(2, 8)
for horizon in horizons:
    print(f"###### Horizon {horizon} #####")
    total_scores = []
    for test_season in reversed(range(2018, 2025)):
        season_diff = test_season - df_matches_all["season"]
        df_matches = df_matches_all[(season_diff < horizon) & (season_diff > 0)]
        df_long = prepare_df_long(df_matches)
        model = PoissonModel()
        model.fit(df_long)

        df_matches_test = df_matches_all[(df_matches_all["season"] == test_season) & (df_matches_all["league"] == "bl1")]
        score = 0
        for _, row in df_matches_test.iterrows():
            home_team = row["host_name"]
            away_team = row["guest_name"]
            correct_result = (row["host_goals"], row["guest_goals"])
            pred = model.predict(home_team, away_team)
            score += Prediction._scoring_rule(pred.max_util_scoreline(), correct_result)

        print(f"Test season {test_season}: {score}")
        total_scores.append(score)

    print("Mean score:", np.mean(total_scores))
    print("Std deviation:", np.std(total_scores))

###### Horizon 2 #####
Test season 2024: 349
Test season 2023: 346
Test season 2022: 357
Test season 2021: 345
Test season 2020: 360
Test season 2019: 357
Test season 2018: 331
Mean score: 349.2857142857143
Std deviation: 9.23834069384271
###### Horizon 3 #####
Test season 2024: 382
Test season 2023: 382
Test season 2022: 426
Test season 2021: 373
Test season 2020: 385
Test season 2019: 405
Test season 2018: 378
Mean score: 390.14285714285717
Std deviation: 17.348763409440306
###### Horizon 4 #####
Test season 2024: 407
Test season 2023: 375
Test season 2022: 418
Test season 2021: 368
Test season 2020: 379
Test season 2019: 382
Test season 2018: 389
Mean score: 388.2857142857143
Std deviation: 16.6794508792302
###### Horizon 5 #####
Test season 2024: 388
Test season 2023: 379
Test season 2022: 393
Test season 2021: 377
Test season 2020: 369
Test season 2019: 412
Test season 2018: 402
Mean score: 388.57142857142856
Std deviation: 13.92692297937593
###### Horizon 6 #####
Test season 2024

## Poisson with correction

In [11]:
from scipy.optimize import minimize
from scipy.stats import poisson

In [12]:
def dixon_cols_tau(x: np.ndarray, y: np.ndarray, lam_h: np.ndarray, lam_a: np.ndarray, rho: float) -> np.ndarray:
    tau = np.ones_like(x, dtype=float)
    mask_00 = (x == 0) & (y == 0)
    mask_01 = (x == 0) & (y == 1)
    mask_10 = (x == 1) & (y == 0)
    mask_11 = (x == 1) & (y == 1)

    tau[mask_00] = 1 - rho * lam_h[mask_00] * lam_a[mask_00]
    tau[mask_01] = 1 + rho * lam_h[mask_01]
    tau[mask_10] = 1 + rho * lam_a[mask_10]
    tau[mask_11] = 1 - rho

    return tau


class DixonColesMLE:
    def __init__(self, xi=0.0, ridge=1e-4) -> None:
        self.xi = float(xi)
        self.ridge = float(ridge)
        self.team_index = {}
        self.teams = None
        self.theta_ = None

    def _unpack(self, theta):
        T = len(self.teams)
        att = np.zeros(T)
        deff = np.zeros(T)
        att[:-1] = theta[:T-1]
        deff[:-1] = theta[T-1:2*T-2]
        att[-1] = -att[:-1].sum()
        deff[-1] = -deff[:-1].sum()
        mu = theta[-2]
        rho = theta[-1]
        return att, deff, mu, rho
    
    def _pack0(self, T):
        return np.concat([np.zeros(T-1), np.zeros(T-1), np.array([0.2, 0.0])])
    
    def _nll(self, theta, df):
        att, deff, mu, rho = self._unpack(theta)

        dt = pd.to_datetime(df["datetime"])
        age = (dt.max() - dt).dt.days.values.astype(float)
        weights = np.exp(-self.xi * age)

        hi = df["host_name"].map(self.team_index)
        ai = df["guest_name"].map(self.team_index)

        lam_h = np.exp(mu + att[hi] - deff[ai])
        lam_a = np.exp(deff[ai] - att[hi])

        log_pois_home = poisson.logpmf(df["host_goals"].values, lam_h)
        log_pois_away = poisson.logpmf(df["guest_goals"].values, lam_a)

        dc_tau = dixon_cols_tau(df["host_goals"].values, df["guest_goals"].values, lam_h, lam_a, rho)
        dc_tau = np.maximum(dc_tau, 1e-12)  # avoid log(0)
        log_dc_tau = np.log(dc_tau)

        nll = -np.sum((log_pois_home + log_pois_away + log_dc_tau) * weights)

        nll += self.ridge * (np.sum(att**2) + np.sum(deff**2) + mu**2 + rho**2)
        return nll
    
    def fit(self, df, verbose=False):
        self.teams = pd.Index(sorted(set(df["host_name"])))
        self.team_index = {t: i for i, t in enumerate(self.teams)}
        T = len(self.teams)

        theta0 = self._pack0(T)
        self._nll(theta0, df)
        res = minimize(self._nll, theta0, args=(df,), method="L-BFGS-B")
        if verbose:
            print(res)
        
        if not res.success:
            raise RuntimeError("Optimization failed")
        
        self.theta_ = res.x
        return self
    
    def params(self):
        if self.theta_ is None:
            raise RuntimeError("Fit the model first.")
        att, deff, mu, rho = self._unpack(self.theta_)
        out = pd.DataFrame({
            "team": self.teams,
            "attack": att,
            "defense": deff
        }).sort_values("team").reset_index(drop=True)
        out["home_advantage_mu"] = mu
        out["rho"] = rho
        return out
    
    def predict(self, home_name, guest_name, max_goals=4): 
        if self.theta_ is None:
            raise RuntimeError("Fit the model first.")
        att, deff, mu, _ = self._unpack(self.theta_)
        try:
            hi = self.team_index[home_name]
            deff_h = deff[hi]
            att_h = att[hi]
        except KeyError:
            deff_h = att_h = 0.5

        try:
            ai = self.team_index[guest_name]
            deff_a = deff[ai]
            att_a = att[ai]
        except KeyError:
            deff_a = att_a = 0.5

        lam_h = np.exp(mu + att_h - deff_a)
        lam_a = np.exp(att_a - deff_h)

        P = np.zeros((max_goals+1, max_goals+1))
        for i in range(max_goals+1):
            # log Poisson for stability; exponentiate at the end
            lpi = poisson.logpmf(i, lam_h)
            for j in range(max_goals+1):
                lpj = poisson.logpmf(j, lam_a)
                tau = dixon_cols_tau(i, j, lam_h, lam_a, self._unpack(self.theta_)[3])
                P[i, j] = np.exp(lpi + lpj) * tau
        P /= P.sum()
        return Prediction(P)

In [ ]:
df_matches_all = pd.read_pickle("data/df_matches_all.pickle")
total_scores = []
for test_season in reversed(range(2018, 2025)):
    season_diff = test_season - df_matches_all["season"]
    df_matches = df_matches_all[(season_diff < 6) & (season_diff > 0) & (df_matches_all["league"] == "bl1")]
    model = DixonColesMLE(xi=0.003, ridge=1e-5)
    model.fit(df_matches)

    df_matches_test = df_matches_all[(df_matches_all["season"] == test_season) & (df_matches_all["league"] == "bl1")]
    score = 0
    for _, row in tqdm.tqdm(df_matches_test.iterrows()):
        home_team = row["host_name"]
        away_team = row["guest_name"]
        correct_result = (row["host_goals"], row["guest_goals"])
        pred = model.predict(home_team, away_team)
        score += Prediction._scoring_rule(pred.max_util_scoreline(), correct_result)

    print(f"Test season {test_season}: {score}")
    total_scores.append(score)

print("Mean score:", np.mean(total_scores))
print("Std deviation:", np.std(total_scores))

306it [00:01, 182.61it/s]


Test season 2024: 316
